# Partie I – MLP et ingénierie PyTorch
**Projet de fin de module – Deep Learning | EMSI Casablanca 2025–2026**

---
### Objectifs
- Comprendre `nn.Module`, paramètres, gradients, `state_dict`, device
- Préparer correctement des données tabulaires réelles
- Implémenter deux versions d'un MLP (`nn.Sequential` et classe personnalisée)
- Comparer trois stratégies d'initialisation
- Sauvegarder / recharger le meilleur modèle
- Évaluer : accuracy, précision, rappel, F1, matrice de confusion
- Répondre à la question de synthèse

**Dataset utilisé :** Wine Quality Red (UCI) — classification supervisée sur données tabulaires

---
## 0. Installation et imports

In [2]:
# Installations (décommenter si nécessaire)
# !pip install torch scikit-learn pandas matplotlib seaborn --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')

ModuleNotFoundError: No module named 'torch'

---
## 1. Concepts fondamentaux PyTorch

### 1.1 `nn.Module` — structure d'un modèle

`nn.Module` est la classe de base de tout réseau de neurones en PyTorch.  
Elle impose deux méthodes :
- `__init__` : déclaration des couches et des paramètres
- `forward(x)` : calcul de la passe avant (propagation)

PyTorch gère automatiquement :
- le suivi des paramètres apprenables (`requires_grad=True`)
- le graphe de calcul pour la rétropropagation
- le placement sur CPU/GPU

### 1.2 Passe avant et rétropropagation

```
Entrée x
  → h¹ = ReLU(W¹x + b¹)      [couche cachée 1]
  → h² = ReLU(W²h¹ + b²)     [couche cachée 2]
  → ŷ  = softmax(W³h² + b³)  [sortie]
  → L  = CrossEntropy(ŷ, y)  [perte]
  → loss.backward()           [calcul des gradients ∂L/∂W]
  → optimizer.step()          [mise à jour des poids]
```

### 1.3 `state_dict` vs `parameters()`

| Concept | Description |
|---|---|
| `model.parameters()` | Tenseurs vivants avec gradient — utilisés par l'optimiseur |
| `model.state_dict()` | Dictionnaire des valeurs seules — utilisé pour sauvegarder |
| `model.named_parameters()` | Idem `parameters()` mais avec le nom de chaque tenseur |

In [ ]:
# Démonstration : un module minimal pour comprendre le mécanisme
class ModuleDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.couche = nn.Linear(3, 2)

    def forward(self, x):
        return self.couche(x)

demo = ModuleDemo()

print('=== Paramètres nommés ===')
for nom, param in demo.named_parameters():
    print(f'  {nom:25s} forme={param.shape}  requires_grad={param.requires_grad}')

print('\n=== State dict ===')
for cle, val in demo.state_dict().items():
    print(f'  {cle:25s} {val}')

x_test = torch.tensor([[1.0, 2.0, 3.0]])
sortie = demo(x_test)
print(f'\nSortie pour x={x_test.tolist()} : {sortie}')

---
## 2. Préparation des données

**Dataset : Wine Quality Red (UCI)**  
- 1599 échantillons, 11 features physico-chimiques  
- Cible : qualité du vin (score 3 à 8 → classification multi-classes)

Pipeline :
1. Chargement et nettoyage
2. Encodage de la cible
3. Découpage stratifié train / val / test (70/15/15)
4. Normalisation (StandardScaler — ajusté uniquement sur le train)
5. Conversion en tenseurs et DataLoaders

> **Erreur critique à éviter** : ajuster le scaler sur le dataset complet fait fuiter la distribution du test dans l'entraînement.

In [ ]:
# Chargement
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
df = pd.read_csv(url, sep=';')
# Option hors-ligne : df = pd.read_csv('winequality-red.csv', sep=';')

print(f'Dimensions : {df.shape}')
print(f'Valeurs manquantes : {df.isnull().sum().sum()}')
print(f'\nDistribution de la cible :')
print(df['quality'].value_counts().sort_index())
df.head()

In [ ]:
# Distribution des features
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='#7F77DD', alpha=0.7, edgecolor='white')
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('Fréquence', fontsize=7)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distribution des features – Wine Quality Red', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('distribution_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pipeline complet
df = df.dropna()

X = df.drop('quality', axis=1).values
y_raw = df['quality'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

n_classes  = len(le.classes_)
n_features = X.shape[1]
print(f'Features     : {n_features}')
print(f'Classes      : {n_classes}  (valeurs originales : {le.classes_})')
print(f'Échantillons : {len(X)}')

# Découpage stratifié 70 / 15 / 15
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)
print(f'\nTrain : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')

# Normalisation
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# DataLoaders
def to_loader(X, y, batch_size=64, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train)
val_loader   = to_loader(X_val,   y_val,   shuffle=False)
test_loader  = to_loader(X_test,  y_test,  shuffle=False)

print('DataLoaders créés.')

---
## 3. Deux implémentations du MLP

### Comparaison

| | `nn.Sequential` | Classe personnalisée |
|---|---|---|
| Rapidité d'écriture | Élevée | Modérée |
| Logique conditionnelle dans `forward` | Non | Oui |
| Connexions résiduelles | Non | Oui |
| Comportement train/eval différencié | Limité | Complet |
| Méthodes personnalisées | Non | Oui |

In [ ]:
N_IN  = n_features
N_H   = 128
N_OUT = n_classes
DROP  = 0.3

# --- Version A : nn.Sequential ---
model_seq = nn.Sequential(
    nn.Linear(N_IN, N_H),
    nn.ReLU(),
    nn.Dropout(DROP),
    nn.Linear(N_H, N_H),
    nn.ReLU(),
    nn.Dropout(DROP),
    nn.Linear(N_H, N_OUT)
)

# --- Version B : classe personnalisée ---
class MLP(nn.Module):
    """
    Perceptron multicouche pour classification tabulaire.
    Architecture : Linear -> ReLU -> Dropout -> Linear -> ReLU -> Dropout -> Linear
    Retourne des logits bruts (CrossEntropyLoss applique softmax en interne).
    """
    def __init__(self, n_in, n_h, n_out, dropout=0.3):
        super().__init__()
        self.fc1  = nn.Linear(n_in, n_h)
        self.fc2  = nn.Linear(n_h,  n_h)
        self.fc3  = nn.Linear(n_h,  n_out)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.drop(self.relu(self.fc1(x)))
        x = self.drop(self.relu(self.fc2(x)))
        return self.fc3(x)

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model_cls = MLP(N_IN, N_H, N_OUT, DROP)

print('=== Résumé architecture (classe) ===')
print(model_cls)
print(f'\nParamètres entraînables : {model_cls.n_params():,}')

# Vérification de la cohérence des deux versions
x_demo = torch.randn(4, N_IN)
print(f'\nSortie Sequential   : {model_seq(x_demo).shape}')
print(f'Sortie Classe perso : {model_cls(x_demo).shape}')

---
## 4. Inspection des paramètres avec `named_parameters()` et `state_dict()`

In [ ]:
print('=== named_parameters() ===')
print(f'{"Nom":<20} {"Forme":<28} {"Numel":>8}  grad')
print('-' * 68)
for nom, param in model_cls.named_parameters():
    print(f'{nom:<20} {str(param.shape):<28} {param.numel():>8,}  {param.requires_grad}')

total = sum(p.numel() for p in model_cls.parameters() if p.requires_grad)
print(f'\nTotal paramètres entraînables : {total:,}')

print('\n=== state_dict() (clés et formes) ===')
for cle, val in model_cls.state_dict().items():
    print(f'  {cle:<20}  {val.shape}')

---
## 5. Trois stratégies d'initialisation

| Stratégie | Formule | Analyse |
|---|---|---|
| Gaussienne | `W ~ N(0, 0.01²)` | Gradients évanescants si std trop faible |
| Constante  | `W = 0.1` | Brise mal la symétrie — tous les neurones identiques, mêmes gradients |
| Xavier     | `std = sqrt(2 / (fan_in + fan_out))` | Variance stable à travers les couches — recommandée avec ReLU |

In [ ]:
def appliquer_init(model, strategie='xavier'):
    """Applique une stratégie d'initialisation à tous les nn.Linear."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if strategie == 'gaussienne':
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
            elif strategie == 'constante':
                nn.init.constant_(m.weight, 0.1)
            elif strategie == 'xavier':
                nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model

model_gauss  = appliquer_init(MLP(N_IN, N_H, N_OUT), 'gaussienne')
model_const  = appliquer_init(MLP(N_IN, N_H, N_OUT), 'constante')
model_xavier = appliquer_init(MLP(N_IN, N_H, N_OUT), 'xavier')

# Visualiser la distribution des poids initiaux
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
configs = [
    (model_gauss,  'Gaussienne (std=0.01)', '#7F77DD'),
    (model_const,  'Constante (0.1)',       '#D85A30'),
    (model_xavier, 'Xavier uniforme',       '#1D9E75'),
]
for ax, (m, titre, couleur) in zip(axes, configs):
    poids = m.fc1.weight.detach().numpy().flatten()
    ax.hist(poids, bins=40, color=couleur, alpha=0.8, edgecolor='white')
    ax.set_title(titre)
    ax.set_xlabel('Valeur du poids')
    ax.set_ylabel('Fréquence')
    ax.axvline(poids.mean(), color='black', linestyle='--', linewidth=1,
               label=f'μ={poids.mean():.4f}\nσ={poids.std():.4f}')
    ax.legend(fontsize=8)

plt.suptitle("Distribution des poids fc1 selon l'initialisation", fontsize=12)
plt.tight_layout()
plt.savefig('distribution_poids_init.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Entraînement comparatif des trois initialisations

In [ ]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS  = 80
LR      = 1e-3
CRITERE = nn.CrossEntropyLoss()

print(f'Device utilisé : {device}\n')

def entrainer_modele(model, train_loader, val_loader, epochs, lr, device, nom=''):
    """Boucle d'entraînement complète. Retourne le modèle et l'historique de perte."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    historique = {'train': [], 'val': []}

    for epoch in range(1, epochs + 1):
        model.train()
        pt = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            perte = CRITERE(model(X_b), y_b)
            perte.backward()
            optimizer.step()
            pt += perte.item()
        pt /= len(train_loader)

        model.eval()
        pv = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                pv += CRITERE(model(X_b.to(device)), y_b.to(device)).item()
        pv /= len(val_loader)

        historique['train'].append(pt)
        historique['val'].append(pv)

        if epoch % 20 == 0:
            print(f'  [{nom}] Époque {epoch:3d}/{epochs} — Train: {pt:.4f} | Val: {pv:.4f}')

    return model, historique

print('--- Gaussienne ---')
model_gauss, hist_gauss = entrainer_modele(
    model_gauss, train_loader, val_loader, EPOCHS, LR, device, 'Gaussienne')
print('--- Constante ---')
model_const, hist_const = entrainer_modele(
    model_const, train_loader, val_loader, EPOCHS, LR, device, 'Constante')
print('--- Xavier ---')
model_xavier, hist_xavier = entrainer_modele(
    model_xavier, train_loader, val_loader, EPOCHS, LR, device, 'Xavier')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
couleurs = {'Gaussienne': '#7F77DD', 'Constante': '#D85A30', 'Xavier': '#1D9E75'}
hists    = [hist_gauss, hist_const, hist_xavier]

for (nom, couleur), hist in zip(couleurs.items(), hists):
    axes[0].plot(hist['train'], label=nom, color=couleur)
    axes[1].plot(hist['val'],   label=nom, color=couleur)

axes[0].set_title('Perte entraînement')
axes[1].set_title('Perte validation')
for ax in axes:
    ax.set_xlabel('Époque')
    ax.set_ylabel('Cross-entropie')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Comparaison des stratégies d'initialisation", fontsize=13)
plt.tight_layout()
plt.savefig('comparaison_initialisations.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Sauvegarde et rechargement du meilleur modèle

On ré-entraîne le modèle Xavier (meilleure initialisation) avec early stopping et `ReduceLROnPlateau`.

In [ ]:
CHEMIN    = 'meilleur_mlp.pt'
EPOCHS_F  = 200
PATIENCE  = 20

model_final = appliquer_init(MLP(N_IN, N_H, N_OUT, DROP), 'xavier').to(device)
optimizer   = optim.Adam(model_final.parameters(), lr=LR, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

meilleure_val = float('inf')
patience_cpt  = 0
hist_final    = {'train': [], 'val': []}

for epoch in range(1, EPOCHS_F + 1):
    model_final.train()
    pt = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        perte = CRITERE(model_final(X_b), y_b)
        perte.backward()
        optimizer.step()
        pt += perte.item()
    pt /= len(train_loader)

    model_final.eval()
    pv = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            pv += CRITERE(model_final(X_b.to(device)), y_b.to(device)).item()
    pv /= len(val_loader)
    scheduler.step(pv)

    hist_final['train'].append(pt)
    hist_final['val'].append(pv)

    if pv < meilleure_val:
        meilleure_val = pv
        patience_cpt  = 0
        torch.save(model_final.state_dict(), CHEMIN)
    else:
        patience_cpt += 1
        if patience_cpt >= PATIENCE:
            print(f'Early stopping à l\'époque {epoch}.')
            break

    if epoch % 30 == 0:
        print(f'Époque {epoch:3d} — Train: {pt:.4f} | Val: {pv:.4f} | Meilleure: {meilleure_val:.4f}')

print(f'\nMeilleure perte validation : {meilleure_val:.4f}')
print(f'Modèle sauvegardé dans     : {CHEMIN}')

# Rechargement
model_charge = MLP(N_IN, N_H, N_OUT, DROP)
model_charge.load_state_dict(torch.load(CHEMIN, map_location='cpu'))
model_charge.eval()
print('\nModèle rechargé avec succès.')
print(f'Clés du state_dict : {list(model_charge.state_dict().keys())}')

---
## 8. Gestion du device CPU / GPU

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device sélectionné : {device}')

model_gpu = MLP(N_IN, N_H, N_OUT).to(device)
X_demo    = torch.randn(8, N_IN).to(device)

dev_modele  = next(model_gpu.parameters()).device
dev_donnees = X_demo.device

print(f'Device modèle   : {dev_modele}')
print(f'Device données  : {dev_donnees}')

assert str(dev_modele) == str(dev_donnees), 'ERREUR : device mismatch !'
print('Cohérence vérifiée — aucun conflit de device.')

with torch.no_grad():
    sortie = model_gpu(X_demo)
print(f'Sortie forme : {sortie.shape}  — device : {sortie.device}')

---
## 9. Évaluation sur le jeu de test

Métriques calculées :
- **Accuracy** : proportion de prédictions correctes
- **Précision** : parmi les prédits positifs, combien sont vrais positifs
- **Rappel** : parmi les vrais positifs, combien sont détectés
- **F1-score** : moyenne harmonique précision/rappel
- **Matrice de confusion** : visualisation des erreurs par classe

In [ ]:
def evaluer_modele(model, loader, device, noms_classes=None):
    model.eval()
    predictions, verites = [], []

    with torch.no_grad():
        for X_b, y_b in loader:
            logits = model(X_b.to(device))
            preds  = logits.argmax(dim=1).cpu().numpy()
            predictions.extend(preds)
            verites.extend(y_b.numpy())

    preds_arr = np.array(predictions)
    verit_arr = np.array(verites)

    acc          = accuracy_score(verit_arr, preds_arr)
    prec, rec, f1, _ = precision_recall_fscore_support(
        verit_arr, preds_arr, average='weighted', zero_division=0
    )

    print('=' * 55)
    print(f'  Exactitude (accuracy) : {acc:.4f}')
    print(f'  Précision (weighted)  : {prec:.4f}')
    print(f'  Rappel    (weighted)  : {rec:.4f}')
    print(f'  F1-score  (weighted)  : {f1:.4f}')
    print('=' * 55)

    print('\nRapport détaillé par classe :')
    print(classification_report(verit_arr, preds_arr,
                                 target_names=noms_classes, zero_division=0))

    mc = confusion_matrix(verit_arr, preds_arr)
    plt.figure(figsize=(7, 6))
    sns.heatmap(mc, annot=True, fmt='d', cmap='Purples',
                xticklabels=noms_classes, yticklabels=noms_classes,
                linewidths=0.5, linecolor='white')
    plt.title('Matrice de confusion — Jeu de test', fontsize=13)
    plt.xlabel('Prédit', fontsize=11)
    plt.ylabel('Réel', fontsize=11)
    plt.tight_layout()
    plt.savefig('matrice_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()

    return acc, prec, rec, f1

noms_classes = [f'qualité {c}' for c in le.classes_]
model_charge = model_charge.to(device)
acc, prec, rec, f1 = evaluer_modele(model_charge, test_loader, device, noms_classes)

In [ ]:
# Courbes finales d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_final['train'], color='#7F77DD', label='Entraînement')
axes[0].plot(hist_final['val'],   color='#D85A30', label='Validation')
axes[0].set_title('Courbe de perte (Cross-entropie)')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Perte')
axes[0].legend()
axes[0].grid(alpha=0.3)

def acc_loader(model, loader, device):
    model.eval()
    preds, verites = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            logits = model(X_b.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            verites.extend(y_b.numpy())
    return accuracy_score(verites, preds)

acc_train = acc_loader(model_charge, train_loader, device)
acc_val   = acc_loader(model_charge, val_loader,   device)
acc_test  = acc_loader(model_charge, test_loader,  device)

barres = [acc_train, acc_val, acc_test]
couleurs_barres = ['#7F77DD', '#D85A30', '#1D9E75']
axes[1].bar(['Train', 'Validation', 'Test'], barres, color=couleurs_barres)
axes[1].set_title('Exactitude par ensemble')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
for i, v in enumerate(barres):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Résultats finaux — MLP (Xavier, Adam, Early Stopping)', fontsize=12)
plt.tight_layout()
plt.savefig('resultats_finaux.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Accuracy — Train: {acc_train:.4f} | Val: {acc_val:.4f} | Test: {acc_test:.4f}')

---
## 10. Question de synthèse

> *Dans quelle mesure un MLP bien paramétré constitue-t-il une solution pertinente pour la classification tabulaire sur un dataset réel, et quelles sont ses principales limites au regard de la structure statistique des données étudiées ?*

### Pertinence

Les données tabulaires n'ont pas de structure spatiale (images) ni temporelle (séquences). Chaque ligne est un vecteur de features indépendantes, ce qui justifie la connexion complète du MLP. Le modèle apprend des frontières de décision non-linéaires qui seraient inaccessibles à une régression logistique.

L'initialisation Xavier converge significativement plus vite et atteint une perte de validation inférieure à la gaussienne et à la constante (voir graphique). L'utilisation d'Adam avec weight decay et d'un scheduler ReduceLROnPlateau stabilise l'entraînement.

### Limites observées

**Déséquilibre de classes** : la matrice de confusion révèle que les classes extrêmes (qualité 3, 4, 8) sont sous-prédites, car sous-représentées dans les données. Correction possible : pondération dans `CrossEntropyLoss(weight=...)` ou rééchantillonnage (SMOTE).

**Interprétabilité** : le MLP est une boîte noire — aucune importance de feature native. Des outils comme SHAP permettraient d'y remédier partiellement.

**Comparaison avec les méthodes ensemblistes** : sur des données tabulaires de taille modeste, Random Forest ou XGBoost surpassent souvent le MLP en précision, sans nécessiter de normalisation ni de réglage fin.

### Conclusion

Un MLP bien paramétré (Xavier, Adam, Dropout, early stopping) est une solution pertinente et compétitive pour la classification tabulaire. Il constitue un excellent point de départ et une référence solide. Son principal avantage est sa généralité : le même paradigme d'apprentissage supervisé s'adapte ensuite aux images (CNN) et aux séquences (RNN), ce qui justifie son étude en premier dans ce module.

In [ ]:
print('=== RÉCAPITULATIF PARTIE I ===')
print(f'Dataset         : Wine Quality Red (UCI)')
print(f'Features        : {N_IN}')
print(f'Classes         : {N_OUT}')
print(f'Architecture    : {N_IN} → {N_H} → {N_H} → {N_OUT}')
print(f'Paramètres      : {model_charge.n_params():,}')
print(f'Initialisation  : Xavier uniforme')
print(f'Optimiseur      : Adam (lr={LR}, weight_decay=1e-4)')
print(f'Régularisation  : Dropout({DROP}) + Early Stopping (patience={PATIENCE})')
print(f'Accuracy test   : {acc_test:.4f}')
print(f'F1-score test   : {f1:.4f}')
print()
print('Fichiers générés :')
for f_name in [
    'meilleur_mlp.pt',
    'distribution_features.png',
    'distribution_poids_init.png',
    'comparaison_initialisations.png',
    'matrice_confusion.png',
    'resultats_finaux.png',
]:
    print(f'  - {f_name}')